# LittleBit quality-recovery runbook (Colab GPU)

This notebook is the actual training route because GitHub-hosted runners in this project have no GPU. Select a Colab **L4 or A100** runtime (BF16 required by Samsung's trainer). A free T4 may not satisfy BF16 and will be rejected rather than silently changing precision.

It pins Samsung LittleBit to `42d658b0c79f76450b34b6a3547462c7cdc3e1a0`, trains at 1.0 effective bits/weight first, evaluates WikiText-2 perplexity, and copies the checkpoint to Google Drive. This is experimental; the notebook does not guarantee that the upstream trainer is compatible with every model architecture.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys, torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Runtime > Change runtime type.'
assert torch.cuda.is_bf16_supported(), 'This run requires BF16: select an L4 or A100 runtime, not a T4.'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
WORK='/content/LittleBit'
PIN='42d658b0c79f76450b34b6a3547462c7cdc3e1a0'
if not os.path.isdir(WORK): subprocess.run(['git','clone','https://github.com/SamsungLabs/LittleBit.git',WORK],check=True)
subprocess.run(['git','-C',WORK,'fetch','--all'],check=True)
subprocess.run(['git','-C',WORK,'checkout',PIN],check=True)
print(subprocess.check_output(['git','-C',WORK,'rev-parse','HEAD'],text=True).strip())

## Install pinned-compatible dependencies

Keep Colab's CUDA-enabled PyTorch rather than downgrading it to the upstream requirements pin. Install the remaining project dependencies and DeepSpeed.

In [ ]:
%cd /content/LittleBit
!python -m pip install -q --upgrade pip
!python -m pip install -q 'transformers>=4.51.0' 'datasets==3.4.1' 'accelerate>=1.6.0' 'tensorboardX' 'pytorch-optimizer>=3.5.0' 'GPUtil>=1.4.0' 'sentencepiece' 'lm-eval==0.4.8' 'deepspeed'
!python -m pip check
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()

## Configure the first recovery run

Start at 1.0 BPW to prioritise quality. The official `main.py` supports `wikitext2`, `SmoothSign`, residuals, ITQ and `l2l_loss_scale`. Dataset preparation is explicitly enabled so the official data loader builds/caches WikiText-2 locally.

In [ ]:
MODEL='Qwen/Qwen2.5-0.5B-Instruct'
EPOCHS='3'
EFF_BIT='1.0'
DATASET='wikitext2'
RUN_NAME='qwen25-05b-littlebit-smoothsign-1bpw'
OUT='/content/drive/MyDrive/littlebit-quality-recovery'
os.makedirs(OUT,exist_ok=True)
print('Output:',OUT)

## Run teacher-guided QAT

The trainer uses BF16, one GPU, and the upstream ZeRO-3 teacher inference config. It can take substantial time and GPU memory. If it fails, preserve the complete traceback; do not claim recovery.

In [ ]:
%cd /content/LittleBit
cmd = [sys.executable, 'main.py', '--model_id', MODEL, '--dataset', DATASET, '--dataset_prepared', 'False', '--save_dir', '/content/littlebit-outputs', '--f_name', RUN_NAME, '--num_train_epochs', EPOCHS, '--per_device_train_batch_size', '1', '--gradient_accumulation_steps', '1', '--lr', '2e-5', '--warmup_ratio', '0.03', '--l2l_loss_scale', '10.0', '--report', 'tensorboard', '--quant_func', 'SmoothSign', '--quant_mod', 'LittleBitLinear', '--residual', 'True', '--eff_bit', EFF_BIT, '--kv_factor', '1.0', '--min_split_dim', '8', '--split_dim', '1024', '--use_itq', 'True', '--itq_n_iter', '50']
print(' '.join(cmd))
log='/content/littlebit-training.log'
with open(log,'w') as f:
    p=subprocess.Popen(cmd,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end='')
        f.write(line); f.flush()
    rc=p.wait()
if rc: raise RuntimeError(f'LittleBit training failed with exit code {rc}; see {log}')
print('Training completed.')

## Evaluate held-out WikiText-2 perplexity

The upstream evaluator loads the saved LittleBit checkpoint and reports WikiText-2 perplexity. Record this alongside the FP teacher baseline; a single short generation sample is not sufficient evidence of recovered quality.

In [ ]:
CKPT=f'/content/littlebit-outputs/{RUN_NAME}'
assert os.path.isdir(CKPT), f'Missing checkpoint: {CKPT}'
eval_cmd=[sys.executable,'eval.py','--model_id',CKPT,'--ppl_task','wikitext2','--zeroshot_task','','--seqlen','2048']
evlog='/content/littlebit-evaluation.log'
with open(evlog,'w') as f:
    p=subprocess.Popen(eval_cmd,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end=''); f.write(line); f.flush()
    rc=p.wait()
if rc: raise RuntimeError(f'Evaluation failed with exit code {rc}; see {evlog}')

## Preserve checkpoint, logs and provenance

Copy the checkpoint and logs to Drive before the Colab session ends. The output contains model weights; keep it private unless you have checked the base model and upstream licence terms.

In [ ]:
import shutil, json, subprocess
dest=os.path.join(OUT,RUN_NAME)
if os.path.exists(dest): shutil.rmtree(dest)
shutil.copytree(CKPT,dest)
shutil.copy2('/content/littlebit-training.log',OUT)
shutil.copy2('/content/littlebit-evaluation.log',OUT)
manifest={'upstream_commit':PIN,'base_model':MODEL,'dataset':DATASET,'epochs':EPOCHS,'effective_bits_per_weight':EFF_BIT,'quant_func':'SmoothSign','quant_mod':'LittleBitLinear','residual':True,'use_itq':True,'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__}
with open(os.path.join(dest,'recovery-run-manifest.json'),'w') as f: json.dump(manifest,f,indent=2)
print('Checkpoint and evidence saved to:',dest)

## What to do with the checkpoint

Download/copy the checkpoint from Drive into the repository's release/artifact workflow only after verifying its contents and licensing. Then run the repository's trained-checkpoint OpenVINO probe in a CPU environment. The probe may need an adapter to the exact upstream `LittleBitModel.from_pretrained` API; do not treat an import/load failure as a quality failure. Compare PyTorch and OpenVINO logits, evaluate perplexity on both runtimes, and review generations manually. Iris Xe is a later, separate validation.